# Build an MCP Server

**What you will build:** your own **MCP server** with FastMCP, exercise its tools in the notebook, and see how any agent — PydanticAI, LangGraph, even Claude Desktop — could consume it. In 18 you *used* someone's MCP server; here you *publish* one. This is the payoff of the whole MCP idea: **write a tool once, use it from everywhere.**

> **Run it:** open in [Google Colab](https://colab.research.google.com/github/ezponda/ai-agents-course/blob/main/courses/python_code/book/18b_build_an_mcp_server.ipynb) (nothing to install), or run locally in Jupyter. Each notebook installs what it needs in its first cell.

In [ ]:
!pip install -q "fastmcp>=2.0,<3.0"

## Why publish a tool over MCP

In 1.2 a tool was a Python function bound to one agent. If a second agent (or a colleague, or Claude Desktop) needs the same capability, they'd re-implement it. An **MCP server** exposes your tool over a standard protocol, so *any* MCP-aware client can call it — no re-implementation, one place to maintain. FastMCP makes a server about as short as the function itself.

## Define a server

`FastMCP("name")` creates the server; `@mcp.tool` publishes a function. Just like PydanticAI, the **type hints become the schema and the docstring becomes the description** the model reads.

In [ ]:
from fastmcp import FastMCP

mcp = FastMCP("Demo Tools")

@mcp.tool
def add(a: int, b: int) -> int:
    """Add two integers and return the sum."""
    return a + b

@mcp.tool
def shout(text: str) -> str:
    """Return the text in upper case with an exclamation mark."""
    return f"{text.upper()}!"

print("server defined with tools: add, shout")

## Exercise it in-memory (no server process needed)

FastMCP's `Client` can connect **directly to the server object** — no network, no separate process — which is perfect for a notebook. This is exactly what a real client does over HTTP, just short-circuited. Read the returned value from `result.data`.

In [ ]:
from fastmcp import Client

async with Client(mcp) as client:                     # in-memory transport
    tools = await client.list_tools()
    print("tools the server exposes:")
    for t in tools:
        print(f"  {t.name} — {t.description}")

    result = await client.call_tool("add", {"a": 2, "b": 3})
    print("\nadd(2, 3) ->", result.data)             # .data is the deserialized Python value -> 5

That's a full MCP round-trip: list the tools, call one by name with arguments, read the result — the same protocol from 0.0, now spoken over MCP. (In a notebook use top-level `await`; never `asyncio.run()`, which errors inside the running loop.)

## Run it as a real HTTP server

To let *other* processes reach it, run the server over HTTP. `mcp.run()` blocks the kernel, so we write it to a file and run it in a terminal (not inline in Colab).

In [ ]:
%%writefile server.py
from fastmcp import FastMCP

mcp = FastMCP("Demo Tools")

@mcp.tool
def add(a: int, b: int) -> int:
    """Add two integers."""
    return a + b

if __name__ == "__main__":
    mcp.run(transport="http", host="127.0.0.1", port=8000)   # serves at http://127.0.0.1:8000/mcp

Run it in a terminal:

```bash
pip install "fastmcp>=2.0,<3.0"
python server.py        # or: fastmcp run server.py:mcp --transport http --port 8000
```

```{note}
In Colab, `mcp.run()` blocks the single kernel and the port isn't public — so run the server locally or on a host, exactly like the FastAPI service in 30. The in-memory `Client(mcp)` cell above is how you test a server *inside* a notebook.
```

## Consume it from an agent

Once the server runs over HTTP, connecting is exactly notebook 18 — point an `MCPToolset` at the URL and the agent uses your tools like any others:

```python
from pydantic_ai import Agent
from pydantic_ai.mcp import MCPToolset

agent = Agent(model, toolsets=[MCPToolset("http://127.0.0.1:8000/mcp")])
async with agent:
    print((await agent.run("What is 7 plus 5?")).output)
```

The same URL works from a LangGraph agent (via `langchain-mcp-adapters`, 18) or from Claude Desktop's config. That's the point: one server, many consumers.

::::{dropdown} 🛠️ Try it yourself
:color: secondary

1. **Add a real tool.** Publish `live_weather` from 1.2 (the `httpx` + wttr.in one) as an `@mcp.tool`, then call it through the in-memory `Client`.
2. **Read the schema.** Print `tools[0].inputSchema` to see the JSON schema FastMCP generated from your type hints — the same shape you wrote by hand in 0.3.
3. **Wire it end-to-end.** Run `server.py` locally and connect to it with the `MCPToolset` snippet above.
::::

**What's next:** you can now both consume (18) and publish (here) tools over MCP. In **19** we build the habit of debugging agents — the last skill of Block 1.